In [0]:
%run ../delta_function

In [0]:
#bibliothèques à importer
import pandas as pd 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, floor
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import col, expr
from pyspark.sql.types import FloatType
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev' #dev
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'parameters'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
catalog_prd = f"""mal_maite_{current_environment}"""

In [0]:
nogent1_features = spark.table(f"{catalog_prd}.nogent1.features") 
nogent2_features = spark.table(f"{catalog_prd}.nogent2.features") 
rouen1_features = spark.table(f"{catalog_prd}.rouen1.features") 
strasbourg2_features = spark.table(f"{catalog_prd}.strasbourg2.features") 
prouvy1_features = spark.table(f"{catalog_prd}.prouvy1.features") 
polisy1_features = spark.table(f"{catalog_prd}.polisy1.features") 
buzau1_features = spark.table(f"{catalog_prd}.buzau1.features")
bolelemi1_features = spark.table(f"{catalog_prd}.bolelemi1.features")

In [0]:
df_features = nogent1_features.unionByName(nogent2_features).unionByName(rouen1_features).unionByName(strasbourg2_features).unionByName(prouvy1_features).unionByName(polisy1_features).unionByName(buzau1_features).unionByName(bolelemi1_features)

df_features_filtered = df_features.filter(
    (F.col("is_controllable") == True)
    & (F.col("deleted") == False)
)

display(df_features_filtered)

In [0]:
# Colonnes à convertir de decimal -> float
decimal_cols = ["min_value", "max_value", "step"]

# Conversion
df_features_converted = df_features_filtered.select(
    *[
        col(c).cast(FloatType()).alias(c) if c in decimal_cols else col(c)
        for c in df_features_filtered.columns
    ]
)

## Bornes dynamiques

`min_value` / `max_value` ne sont plus repris tels quels des tables `features` : ils sont
recalcules sur la population de reference (percentile 1 / percentile 99), en repliquant
`derive_bounds()` du projet d'inference (DM-6232 / DM-6452).

Population de reference (DM-6452) : `deleted = false`, `batch_status` dans
('finished', 'in_progress'), moins les `batch_id` listes dans `mttts_ignore`.

Les valeurs statiques des tables `features` restent le repli quand la feature est absente
de la master table ou n'y a aucune donnee valide -- meme regle que cote optimiseur.

In [ ]:
import math

# Constantes reprises de files/inference/src/optimizers/bounds.py : toute divergence ici
# ferait afficher au rapport des bornes que l'optimiseur n'utilise pas.
BOUND_FOR_MIN_PCTL = 0.01
BOUND_FOR_MAX_PCTL = 0.99
ACTIVE_BATCH_STATUSES = ("finished", "in_progress")

# Memes sites que les tables features chargees plus haut.
sites = ["nogent1", "nogent2", "rouen1", "strasbourg2", "prouvy1", "polisy1", "buzau1", "bolelemi1"]

In [ ]:
import hashlib

# derive_bounds() n'est pas appelee mais recopiee plus bas : rien ne garantit donc que
# notre copie reste fidele si la DS fait evoluer son calcul. On garde l'empreinte du
# fichier source et on s'arrete des qu'elle bouge -- une copie perimee ne leverait
# aucune erreur, elle afficherait juste des bornes que l'optimiseur n'utilise plus.
#
# Au premier run, la cellule affiche l'empreinte a reporter ici.
BOUNDS_PY_SHA256 = ""

bounds_py = f"/Workspace/Shared/maite/{sites[0]}/mal_maite_code/files/inference/src/optimizers/bounds.py"

try:
    current_sha = hashlib.sha256(open(bounds_py, "rb").read()).hexdigest()
except FileNotFoundError:
    current_sha = None
    print(f"ATTENTION : {bounds_py} introuvable, la derive ne peut pas etre verifiee")

if current_sha and not BOUNDS_PY_SHA256:
    print(f"Empreinte a reporter dans BOUNDS_PY_SHA256 : {current_sha}")
elif current_sha and current_sha != BOUNDS_PY_SHA256:
    raise RuntimeError(
        f"bounds.py a change cote inference (empreinte {current_sha}).\n"
        f"Comparer derive_bounds / round_bound / snap_to_step avec la copie de ce notebook, "
        f"reporter les evolutions, puis mettre a jour BOUNDS_PY_SHA256."
    )
else:
    print(f"bounds.py inchange depuis la recopie ({bounds_py})")

In [ ]:
def build_reference_population(site):
    """
    Reduit la master table du site aux batchs autorises a servir de reference :
    non supprimes, finished ou in_progress, et absents de mttts_ignore.

    Replique RecommendationInputTables.build_reference_population. La comparaison des
    batch_id se fait en chaine des deux cotes, comme cote DS, pour ne pas dependre du
    type de la colonne.

    Rien n'est rattrape ici : une colonne de filtrage manquante fait echouer la lecture
    Spark, et une table mttts_ignore absente fait echouer la cellule. Cote inference ces
    deux cas levent une ValueError depuis la correction du 24/09, parce que continuer
    reviendrait a calculer les bornes sur des batchs que le metier a ecartes. Une table
    mttts_ignore *vide* est en revanche le cas normal d'une ligne sans batch exclu.
    """
    mt = spark.table(f"mal_maite_{site}_{current_environment}.gold.master_table")

    reference = mt.filter(
        (F.col("deleted") == False) & (F.col("batch_status").isin(*ACTIVE_BATCH_STATUSES))
    )
    n_active = reference.count()

    ignored = (
        spark.table(f"{catalog_prd}.{site}.mttts_ignore")
        .select(F.col("batch_id").cast("string").alias("ignored_batch_id"))
        .distinct()
    )
    n_ignored = ignored.count()

    if n_ignored:
        reference = (
            reference
            .withColumn("batch_id_str", F.col("batch_id").cast("string"))
            .join(F.broadcast(ignored), F.col("batch_id_str") == F.col("ignored_batch_id"), "left_anti")
            .drop("batch_id_str")
        )

    n_reference = reference.count()
    n_removed = n_active - n_reference
    print(f"{site} : {n_active} batchs actifs -> {n_reference} apres exclusion de "
          f"{n_removed} ligne(s) sur {n_ignored} batch_id ignore(s)")

    # Un ecart de format entre les batch_id des deux tables ne leve aucune erreur : la
    # liste d'exclusion devient juste sans effet, et les batchs ecartes par le metier
    # reviennent peser sur les bornes. On le signale plutot que de le laisser passer.
    if n_ignored and not n_removed:
        print(f"{site} : ATTENTION, aucun des {n_ignored} batch_id ignores ne correspond a un "
              f"batch de la master table, la liste d'exclusion n'a aucun effet")

    return reference

In [ ]:
def compute_site_bounds(site, reference, feature_names):
    """
    Percentile 1 / 99 de chaque feature sur la population de reference, en une seule passe.

    La master table est large (1 colonne = 1 feature) : on la depivote avec stack() puis on
    agrege par (production_line, feature_reference), plutot que d'empiler une agregation par
    feature.

    percentile() exact et surtout pas percentile_approx : c'est la seule variante Spark qui
    interpole comme pandas Series.quantile(), utilise cote optimiseur. Le CAST en double
    reproduit pd.to_numeric(errors="coerce") -- ce qui n'est pas numerique devient nul.
    """
    present = [f for f in feature_names if f in reference.columns]
    if not present:
        print(f"{site} : aucune feature controlable ne correspond a une colonne de la master table")
        return None

    pairs = ", ".join(f"'{f}', CAST(`{f}` AS DOUBLE)" for f in present)

    # production_line est en minuscules dans la master table, en majuscules dans les tables
    # features : sans cet upper() la jointure ne ramene rien et tout part en repli.
    long = reference.select(
        F.upper(F.col("production_line")).alias("production_line"),
        F.expr(f"stack({len(present)}, {pairs}) as (feature_reference, value)"),
    )

    return (
        long.filter(F.col("value").isNotNull())
        .groupBy("production_line", "feature_reference")
        .agg(
            F.expr(f"percentile(value, {BOUND_FOR_MIN_PCTL})").alias("pctl_min"),
            F.expr(f"percentile(value, {BOUND_FOR_MAX_PCTL})").alias("pctl_max"),
            F.count("value").alias("n_rows"),
        )
    )

In [ ]:
feature_names = [
    r.feature_reference
    for r in df_features_converted.select("feature_reference").distinct().collect()
]
print(f"{len(feature_names)} features controlables distinctes")

site_bounds = []
for site in sites:
    bounds = compute_site_bounds(site, build_reference_population(site), feature_names)
    if bounds is not None:
        site_bounds.append(bounds)

dynamic_bounds = reduce(lambda a, b: a.unionByName(b), site_bounds)

if verbose_mode == 'debug':
    display(dynamic_bounds)

In [ ]:
def round_bound(value):
    """Repris a l'identique de bounds.py."""
    rounded = round(float(value), 2)
    return int(rounded) if rounded == int(rounded) else rounded


def snap_to_step(min_val, max_val, step):
    """Repris a l'identique de bounds.py."""
    def _snap(value, fn):
        snapped = fn(round(value / step, 9)) * step
        return int(snapped) if snapped == int(snapped) else round(snapped, 9)

    return _snap(min_val, math.floor), _snap(max_val, math.ceil)


def resolve_bounds(row):
    """
    Arrondi puis calage sur la grille du step, dans cet ordre -- c'est celui de derive_bounds,
    et l'inverser change les valeurs.

    En pandas et non en Spark parce que round() n'arrondit pas pareil dans les deux moteurs
    sur les valeurs a mi-chemin, et que c'est la version Python qui fait foi cote optimiseur.
    Le volume le permet : une ligne par couple (ligne de production x feature).
    """
    if pd.isna(row["n_rows"]) or row["n_rows"] == 0:
        return pd.Series({"min_value": row["min_value"], "max_value": row["max_value"], "source": "static"})

    min_val = round_bound(row["pctl_min"])
    max_val = round_bound(row["pctl_max"])

    step = row["step"]
    if step and step > 0:
        min_val, max_val = snap_to_step(min_val, max_val, step)

    return pd.Series({"min_value": min_val, "max_value": max_val, "source": "dynamic"})


features_pd = df_features_converted.toPandas()
merged = features_pd.merge(
    dynamic_bounds.toPandas(), on=["production_line", "feature_reference"], how="left"
)

resolved = merged.apply(resolve_bounds, axis=1)
n_changed = int(
    (merged["min_value"] != resolved["min_value"]).sum()
    + (merged["max_value"] != resolved["max_value"]).sum()
)
merged["min_value"] = resolved["min_value"].astype(float)
merged["max_value"] = resolved["max_value"].astype(float)

n_dynamic = int((resolved["source"] == "dynamic").sum())
print(f"{n_dynamic} / {len(merged)} bornes recalculees, "
      f"{len(merged) - n_dynamic} en repli sur les valeurs statiques")
print(f"{n_changed} borne(s) modifiee(s) par rapport aux valeurs actuelles")

df_features_with_dynamic_bounds = spark.createDataFrame(merged[df_features_converted.columns])
for c in decimal_cols:
    df_features_with_dynamic_bounds = df_features_with_dynamic_bounds.withColumn(
        c, F.col(c).cast(FloatType())
    )

IMPORT

In [0]:
table_features_min_max = df_features_with_dynamic_bounds.select(
    "production_line",
    "feature_reference",
    "min_value",
    "max_value",
    "is_controllable",
    "deleted",
    "is_categorical",
    "is_stratification",
    "step",
    "activity"
)

In [0]:
current_process= "fact_features_min_max"

In [0]:
target_fact_features_min_max = current_catalog +"."+current_schema+"."+current_process
print(target_fact_features_min_max)

In [0]:
all_columns =  table_features_min_max.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'production_line'
    ,'feature_reference']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    table_features_min_max, 
    target_fact_features_min_max, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )